# Hyperparameter Tuning — XGBoost with Optuna

In [ ]:
import os
import sys
import dill
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from src.utils import evaluate_models

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

project_root = os.getcwd()
print(f'Project root: {project_root}')

In [ ]:
# ── Colab Setup ──────────────────────────────────────────────────────────────
import os, sys

# Option 1 — Clone from GitHub (recommended if repo is pushed)
# !git clone https://github.com/YOUR_USERNAME/power_price_prediction.git
# os.chdir('/content/power_price_prediction')

# Option 2 — Mount Google Drive (if project is uploaded there)
# from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/power_price_prediction')

# ── After choosing one option above, run this ─────────────────────────────────
sys.path.insert(0, os.getcwd())

!pip install -q optuna dill category_encoders xgboost catboost scikit-learn

## 1. Load & Transform Data

In [2]:
from src.components.data_transformation import DataTransformation

train_path = os.path.join(project_root, 'artifact', 'train.csv')
test_path  = os.path.join(project_root, 'artifact', 'test.csv')

transformation = DataTransformation()
train_arr, test_arr, _ = transformation.initiate_data_transformation(train_path, test_path)

X_train, y_train = train_arr[:, :-1], train_arr[:, -1]
X_test,  y_test  = test_arr[:, :-1],  test_arr[:, -1]

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')

ModuleNotFoundError: No module named 'src'

## 2. Optuna Hyperparameter Search

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0.0, 0.5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1.0, 4.0),
    }

    report = evaluate_models(
        X_train, y_train, X_test, y_test,
        models={'XGBoost': XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)},
        params={'XGBoost': params}
    )
    return report['XGBoost']

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nBest R2  : {study.best_value:.4f}')
print(f'Best params:\n{study.best_params}')

## 3. Train Best Model & Evaluate

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

tuned_model = XGBRegressor(**study.best_params, random_state=42, n_jobs=-1, verbosity=0)
tuned_model.fit(X_train, y_train)
pred = tuned_model.predict(X_test)

print(f'R2  : {r2_score(y_test, pred):.4f}')
print(f'MAE : {mean_absolute_error(y_test, pred):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, pred)):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sample_idx = np.random.choice(len(y_test), min(3000, len(y_test)), replace=False)
axes[0].scatter(y_test[sample_idx], pred[sample_idx], alpha=0.3, s=10, color='steelblue')
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
axes[0].plot(lims, lims, 'r--', lw=1.5)
axes[0].set(xlabel='Actual', ylabel='Predicted', title='Actual vs Predicted')

residuals = y_test[sample_idx] - pred[sample_idx]
axes[1].scatter(pred[sample_idx], residuals, alpha=0.3, s=10, color='darkorange')
axes[1].axhline(0, color='red', linestyle='--', lw=1.5)
axes[1].set(xlabel='Predicted', ylabel='Residual', title='Residual Plot')

plt.tight_layout()
plt.show()

## 4. Optuna Visualizations

In [ ]:
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.tight_layout()
plt.show()

In [ ]:
optuna.visualization.matplotlib.plot_param_importances(study)
plt.tight_layout()
plt.show()

## 5. Feature Importances

In [ ]:
fi = pd.Series(tuned_model.feature_importances_).sort_values(ascending=False).head(20)
plt.figure(figsize=(9, 6))
sns.barplot(x=fi.values, y=fi.index.astype(str), palette='viridis', orient='h')
plt.title('Top 20 Feature Importances — XGBoost (tuned)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 6. Save Tuned Model

In [ ]:
output_path = os.path.join(project_root, 'artifact', 'model_tuned.pkl')
with open(output_path, 'wb') as f:
    dill.dump(tuned_model, f)

print(f'Saved to: {output_path}')